In [4]:
import yaml
from ultralytics import YOLO

In [5]:
import yaml
from pathlib import Path

dataset_root = Path("Poles2025/Road_poles_iPhone").resolve()
yaml_path = dataset_root / "data.yaml"

with open(yaml_path) as f:
    cfg = yaml.safe_load(f)

if "Train" in cfg:      cfg["train"] = cfg.pop("Train")
if "Validation" in cfg: cfg["val"]   = cfg.pop("Validation")
if "Test" in cfg:       cfg["test"]  = cfg.pop("Test")

cfg["path"] = str(dataset_root)

with open(yaml_path, "w") as f:
    yaml.dump(cfg, f)

# Quick verify
sample = dataset_root / "data/images/Train/train"
print("Images folder exists:", sample.exists())
print("Sample:", list(sample.iterdir())[0])

Images folder exists: True
Sample: /work/gustavga/Poles2025/Road_poles_iPhone/data/images/Train/train/920916896.jpg


In [8]:
yaml_path = Path("Poles2025/Road_poles_iPhone/data.yaml")

cfg = {
    "path":  "/work/gustavga/Poles2025/Road_poles_iPhone",
    "train": "data/images/Train/train",
    "val":   "data/images/Val/val",
    "test":  "data/images/Test/test",
    "names": {0: "pole"}
}

with open(yaml_path, "w") as f:
    yaml.dump(cfg, f)

# Delete cache
for cache in Path("Poles2025/Road_poles_iPhone").rglob("*.cache"):
    cache.unlink()
    print("deleted:", cache)

print("Done:")
with open(yaml_path) as f:
    print(f.read())

Done:
names:
  0: pole
path: /work/gustavga/Poles2025/Road_poles_iPhone
test: data/images/Test/test
train: data/images/Train/train
val: data/images/Val/val



In [11]:
model = YOLO("yolov8n.pt")
model.train(data=str(yaml_path.resolve()), epochs=100, imgsz=640, device=0)

Ultralytics 8.4.39 🚀 Python-3.12.3 torch-2.11.0+cu130 


ValueError: Invalid CUDA 'device=0' requested. Use 'device=cpu' or pass valid CUDA device(s) if available, i.e. 'device=0' or 'device=0,1,2,3' for Multi-GPU.

torch.cuda.is_available(): False
torch.cuda.device_count(): 1
os.environ['CUDA_VISIBLE_DEVICES']: 0


In [12]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch
print(torch.cuda.is_available())     # should now be True
print(torch.cuda.get_device_name(0)) # should show GPU name

False


RuntimeError: The NVIDIA driver on your system is too old (found version 12090). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver.